# Collections statistics

Descriptive statistics about the **Collections** currently loaded in the Neo4j
database by `2_00_save_collections` (run that notebook first). For each
collection we report whether it is a CellDesigner disease map collection
(`*_DM_CD`) or a BEL knowledge graph (`*_KG_BEL`), and counts of its content.

This notebook is read-only: it only queries the DB and writes summary CSVs to
`RESULTS_DIR/collections_statistics/`. It must be run **after** `2_00` (the
collections must exist) and does not depend on `2_10` (the influence graph).

In [1]:
%run 0_10_load_paths.ipynb

In [2]:
import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session
import pandas as pd

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

## Statistics functions

Each function runs a single Cypher query and returns a `pandas.DataFrame`. The
two collection kinds have different shapes in the DB:

- **CellDesigner collections** (`COVID_DM_CD`, `PD_DM_CD`): one
  `CellDesignerMap` per `CollectionEntry`, each `Model` linked to its elements
  by `HAS_MODEL_ELEMENT`. We count the top-level model-element labels
  (`Species`, `Reaction`, `Modulation`, `Compartment`).
- **BEL collections** (`AD_KG_BEL`, `PD_KG_BEL`, `COVID_KG_BEL`): a single
  `BELModel` per collection, with model elements linked by `HAS_NODE` and
  thematic `Subgraph` nodes linked by `HAS_SUBGRAPH`.

In [4]:
def get_collections_overview(session):
    """One row per collection: kind (CD/BEL), number of entries (maps for CD),
    and total number of model elements / nodes."""
    query = """
        MATCH (c:Collection)-[:HAS_ENTRY]->(e:CollectionEntry)
        OPTIONAL MATCH (e)-[:HAS_OBJ]->(cd:CellDesignerMap)
        OPTIONAL MATCH (e)-[:HAS_OBJ]->(bel:BELModel)
        WITH c, e, cd, bel
        OPTIONAL MATCH (cd)-[:HAS_MODEL]->(:CellDesignerModel)-[:HAS_MODEL_ELEMENT]->(cd_el)
        OPTIONAL MATCH (bel)-[:HAS_NODE]->(bel_node) WHERE NOT bel_node:Subgraph
        RETURN
            c.name AS collection,
            CASE WHEN count(DISTINCT cd) > 0 THEN "CellDesigner" ELSE "BEL" END AS kind,
            count(DISTINCT e) AS n_entries,
            count(DISTINCT cd_el) + count(DISTINCT bel_node) AS n_elements
        ORDER BY collection
    """
    rows = session.execute_query(query)
    df = pd.DataFrame(
        [dict(r) for r in rows],
        columns=["collection", "kind", "n_entries", "n_elements"],
    )
    return df.sort_values("n_elements", ascending=False).reset_index(drop=True)

In [5]:
def get_cd_collections_statistics(session):
    """Per CellDesigner collection: total counts of species, reactions,
    modulations and compartments across all of its maps, plus the number of
    distinct proteins (by `hgnc.symbol` annotation)."""
    query = """
        MATCH (c:Collection)-[:HAS_ENTRY]->(:CollectionEntry)
            -[:HAS_OBJ]->(:CellDesignerMap)-[:HAS_MODEL]->(m:CellDesignerModel)
            -[:HAS_MODEL_ELEMENT]->(e)
        RETURN
            c.name AS collection,
            sum(CASE WHEN e:Species THEN 1 ELSE 0 END) AS species,
            sum(CASE WHEN e:Reaction THEN 1 ELSE 0 END) AS reactions,
            sum(CASE WHEN e:Modulation THEN 1 ELSE 0 END) AS modulations,
            sum(CASE WHEN e:Compartment THEN 1 ELSE 0 END) AS compartments
        ORDER BY collection
    """
    counts = pd.DataFrame(
        [dict(r) for r in session.execute_query(query)],
        columns=["collection", "species", "reactions", "modulations", "compartments"],
    )
    # Distinct proteins by hgnc.symbol annotation (same notion of "protein" the
    # interface analysis in 3_00 uses):
    query = """
        MATCH (c:Collection)-[:HAS_ENTRY]->(:CollectionEntry)
            -[:HAS_ELEMENT_TO_ANNOTATIONS]->(:Mapping)
            -[:HAS_ITEM]->(:Item)-[:HAS_VALUE]->(:Bag)-[:HAS_ITEM]->(a:RDFAnnotation)
        UNWIND a.resources AS resource
        WITH c.name AS collection, resource
        WHERE resource CONTAINS "hgnc.symbol"
        RETURN collection, count(DISTINCT split(resource, ":")[-1]) AS distinct_hgnc_symbols
        ORDER BY collection
    """
    symbols = pd.DataFrame(
        [dict(r) for r in session.execute_query(query)],
        columns=["collection", "distinct_hgnc_symbols"],
    )
    df = counts.merge(symbols, on="collection", how="left")
    return df.sort_values("species", ascending=False).reset_index(drop=True)

In [6]:
def get_cd_species_breakdown(session):
    """Per CellDesigner collection: number of species of each CellDesigner
    species class (Protein, Complex, SimpleMolecule, RNA, Gene, ...)."""
    query = """
        MATCH (c:Collection)-[:HAS_ENTRY]->(:CollectionEntry)
            -[:HAS_OBJ]->(:CellDesignerMap)-[:HAS_MODEL]->(:CellDesignerModel)
            -[:HAS_MODEL_ELEMENT]->(e:Species)
        WITH c.name AS collection, [l IN labels(e) WHERE NOT l IN [
            "BaseNode", "MapElement", "ModelElement", "CellDesignerModelElement", "Species"
        ]][0] AS species_class
        RETURN collection, species_class, count(*) AS n
        ORDER BY collection, n DESC
    """
    df = pd.DataFrame(
        [dict(r) for r in session.execute_query(query)],
        columns=["collection", "species_class", "n"],
    )
    pivot = (
        df.pivot(index="species_class", columns="collection", values="n")
        .fillna(0)
        .astype(int)
    )
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]
    pivot["Total"] = pivot.sum(axis=1)
    return pivot

In [7]:
def get_bel_collections_statistics(session):
    """Per BEL collection: number of model nodes, number of thematic subgraphs,
    and number of (non-structural) relationships among its nodes."""
    query = """
        MATCH (c:Collection)-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(m:BELModel)
        RETURN
            c.name AS collection,
            count { (m)-[:HAS_NODE]->(n) WHERE NOT n:Subgraph } AS n_nodes,
            count { (m)-[:HAS_SUBGRAPH]->(:Subgraph) } AS n_subgraphs
        ORDER BY collection
    """
    nodes = pd.DataFrame(
        [dict(r) for r in session.execute_query(query)],
        columns=["collection", "n_nodes", "n_subgraphs"],
    )
    query = """
        MATCH (c:Collection)-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(m:BELModel)
        MATCH (m)-[:HAS_NODE]->(a)-[r]->(b)<-[:HAS_NODE]-(m)
        WHERE NOT type(r) STARTS WITH "HAS_"
        RETURN c.name AS collection, count(r) AS n_relationships
        ORDER BY collection
    """
    rels = pd.DataFrame(
        [dict(r) for r in session.execute_query(query)],
        columns=["collection", "n_relationships"],
    )
    df = nodes.merge(rels, on="collection", how="left")
    return df.sort_values("n_nodes", ascending=False).reset_index(drop=True)

In [8]:
def get_bel_node_breakdown(session):
    """Per BEL collection: number of nodes of each BEL element class."""
    query = """
        MATCH (c:Collection)-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(m:BELModel)
            -[:HAS_NODE]->(n)
        WHERE NOT n:Subgraph
        WITH c.name AS collection, [l IN labels(n) WHERE NOT l IN [
            "BELModelElement", "BioConcept", "GeneticFlow"
        ]][0] AS node_class
        RETURN collection, node_class, count(*) AS n
        ORDER BY collection, n DESC
    """
    df = pd.DataFrame(
        [dict(r) for r in session.execute_query(query)],
        columns=["collection", "node_class", "n"],
    )
    pivot = (
        df.pivot(index="node_class", columns="collection", values="n")
        .fillna(0)
        .astype(int)
    )
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]
    pivot["Total"] = pivot.sum(axis=1)
    return pivot

In [9]:
def get_bel_relationship_breakdown(session):
    """Per BEL collection: number of (non-structural) relationships of each BEL
    relation type among its nodes. BEL relations fall into causal
    (INCREASES/DECREASES/..., REGULATES, CAUSES_NO_CHANGE), correlative
    (ASSOCIATION, POSITIVE/NEGATIVE_CORRELATION), hierarchical (IS_A),
    equivalence (ORTHOLOGOUS, EQUIVALENT_TO) and transformation (TRANSLATED_TO)
    families. Only the causal family is carried into the influence graph by
    `ig.make_ig_in_db`."""
    query = """
        MATCH (c:Collection)-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(m:BELModel)
        MATCH (m)-[:HAS_NODE]->(a)-[r]->(b)<-[:HAS_NODE]-(m)
        WHERE NOT type(r) STARTS WITH "HAS_"
        RETURN c.name AS collection, type(r) AS relation_type, count(r) AS n
        ORDER BY collection, n DESC
    """
    df = pd.DataFrame(
        [dict(r) for r in session.execute_query(query)],
        columns=["collection", "relation_type", "n"],
    )
    pivot = (
        df.pivot(index="relation_type", columns="collection", values="n")
        .fillna(0)
        .astype(int)
    )
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]
    pivot["Total"] = pivot.sum(axis=1)
    return pivot

In [10]:
def get_bel_endpoint_type_pair_breakdown(session):
    """Per BEL collection: number of (non-structural) relationships for each
    *unordered* pair of endpoint node classes (e.g. Protein - Protein,
    BiologicalProcess - Protein). Endpoint classes are the same primary BEL
    labels used in `get_bel_node_breakdown`; the pair is order-independent, so
    a Protein->BiologicalProcess and a BiologicalProcess->Protein edge are
    counted together."""
    query = """
        MATCH (c:Collection)-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(m:BELModel)
        MATCH (m)-[:HAS_NODE]->(a)-[r]->(b)<-[:HAS_NODE]-(m)
        WHERE NOT type(r) STARTS WITH "HAS_"
        WITH c.name AS collection,
            [l IN labels(a) WHERE NOT l IN ["BELModelElement", "BioConcept", "GeneticFlow", "Subgraph"]][0] AS source_class,
            [l IN labels(b) WHERE NOT l IN ["BELModelElement", "BioConcept", "GeneticFlow", "Subgraph"]][0] AS target_class
        WITH collection, apoc.coll.sort([source_class, target_class]) AS pair
        RETURN collection, pair[0] + " - " + pair[1] AS type_pair, count(*) AS n
        ORDER BY collection, n DESC
    """
    df = pd.DataFrame(
        [dict(r) for r in session.execute_query(query)],
        columns=["collection", "type_pair", "n"],
    )
    pivot = (
        df.pivot(index="type_pair", columns="collection", values="n")
        .fillna(0)
        .astype(int)
    )
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]
    pivot["Total"] = pivot.sum(axis=1)
    return pivot

In [11]:
def get_bel_grouped_endpoint_pair_breakdown(session):
    """Per BEL collection: number of (non-structural) relationships for each
    *unordered* pair of two coarse endpoint groups:

    - **abundance/activity**: abundance-type entities and their activities --
      `Protein`, `Complex`, `Abundance`, `Gene`, `Rna`, `MicroRna`, `Composite`,
      `Variant`, `Activity`.
    - **biologicalprocess/pathology**: the `BioConcept` nodes (BiologicalProcess
      and Pathology).

    Every *other* node type (transformations, modifiers, locations, lists,
    reactions, ...) is excluded, and only relationships whose *both* endpoints
    are in the counted set are tallied (induced subgraph). Because excluded
    nodes drop out, these three rows do **not** sum to the collection's total
    relationship count."""
    query = """
        MATCH (c:Collection)-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(m:BELModel)
        MATCH (m)-[:HAS_NODE]->(a)-[r]->(b)<-[:HAS_NODE]-(m)
        WHERE NOT type(r) STARTS WITH "HAS_"
        WITH c.name AS collection,
            CASE
                WHEN a:BioConcept THEN "biologicalprocess/pathology"
                WHEN a:Protein OR a:Complex OR a:Abundance OR a:Gene OR a:Rna
                    OR a:MicroRna OR a:Composite OR a:Variant OR a:Activity
                    THEN "abundance/activity"
                ELSE null
            END AS source_group,
            CASE
                WHEN b:BioConcept THEN "biologicalprocess/pathology"
                WHEN b:Protein OR b:Complex OR b:Abundance OR b:Gene OR b:Rna
                    OR b:MicroRna OR b:Composite OR b:Variant OR b:Activity
                    THEN "abundance/activity"
                ELSE null
            END AS target_group
        WHERE source_group IS NOT NULL AND target_group IS NOT NULL
        WITH collection, apoc.coll.sort([source_group, target_group]) AS pair
        RETURN collection, pair[0] + " - " + pair[1] AS group_pair, count(*) AS n
        ORDER BY collection, group_pair
    """
    df = pd.DataFrame(
        [dict(r) for r in session.execute_query(query)],
        columns=["collection", "group_pair", "n"],
    )
    pivot = (
        df.pivot(index="group_pair", columns="collection", values="n")
        .fillna(0)
        .astype(int)
    )
    # Fixed row order: same-group abundance, mixed, same-group bioconcept.
    row_order = [
        "abundance/activity - abundance/activity",
        "abundance/activity - biologicalprocess/pathology",
        "biologicalprocess/pathology - biologicalprocess/pathology",
    ]
    pivot = pivot.reindex(row_order).fillna(0).astype(int)
    pivot["Total"] = pivot.sum(axis=1)
    return pivot

In [12]:
def get_bel_degree_data(session):
    """Per BEL node in the counted set, its in- and out-degree over the induced
    subgraph.

    The counted set is the same two groups used by
    `get_bel_grouped_endpoint_pair_breakdown`: **abundance/activity**
    (`Protein`, `Complex`, `Abundance`, `Gene`, `Rna`, `MicroRna`, `Composite`,
    `Variant`, `Activity`) and **biologicalprocess/pathology** (`BioConcept`).
    Degrees count only non-structural relationships (`type` not starting with
    `HAS_`) whose *both* endpoints are in the counted set. Returns one row per
    node: `collection`, `node_group`, `indegree`, `outdegree`."""
    query = """
        MATCH (c:Collection)-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(m:BELModel)
            -[:HAS_NODE]->(n)
        WITH c, n,
            CASE
                WHEN n:BioConcept THEN "biologicalprocess/pathology"
                WHEN n:Protein OR n:Complex OR n:Abundance OR n:Gene OR n:Rna
                    OR n:MicroRna OR n:Composite OR n:Variant OR n:Activity
                    THEN "abundance/activity"
                ELSE null
            END AS node_group
        WHERE node_group IS NOT NULL
        OPTIONAL MATCH (n)<-[ri]-(src)
            WHERE NOT type(ri) STARTS WITH "HAS_"
                AND (src:BioConcept OR src:Protein OR src:Complex OR src:Abundance
                    OR src:Gene OR src:Rna OR src:MicroRna OR src:Composite
                    OR src:Variant OR src:Activity)
        WITH c, n, node_group, count(ri) AS indegree
        OPTIONAL MATCH (n)-[ro]->(tgt)
            WHERE NOT type(ro) STARTS WITH "HAS_"
                AND (tgt:BioConcept OR tgt:Protein OR tgt:Complex OR tgt:Abundance
                    OR tgt:Gene OR tgt:Rna OR tgt:MicroRna OR tgt:Composite
                    OR tgt:Variant OR tgt:Activity)
        WITH c, n, node_group, indegree, count(ro) AS outdegree
        RETURN c.name AS collection, node_group, indegree, outdegree
    """
    return pd.DataFrame(
        [dict(r) for r in session.execute_query(query)],
        columns=["collection", "node_group", "indegree", "outdegree"],
    )

In [13]:
def fit_degree_power_law(degrees):
    """Fit a discrete power law to a 1-D array of node degrees using the
    `powerlaw` package (the Clauset-Shalizi-Newman method): MLE-estimate the
    exponent `alpha` and lower cutoff `x_min`, then compare the power law
    against a lognormal and against an exponential via normalised
    log-likelihood ratios.

    Returns a dict with `alpha`, `x_min`, the tail size `n_tail`, and for each
    alternative a ratio `R` (>0 favours the power law, <0 favours the
    alternative) with its significance `p`. The fitted `powerlaw.Fit` object is
    returned under `_fit` for plotting. Degrees of 0 are dropped (a power law
    lives on k >= x_min >= 1)."""
    import numpy as np
    import powerlaw

    degrees = np.asarray(degrees)
    degrees = degrees[degrees > 0]
    fit = powerlaw.Fit(degrees, discrete=True, verbose=False)
    r_lognormal, p_lognormal = fit.distribution_compare(
        "power_law", "lognormal", normalized_ratio=True
    )
    r_exponential, p_exponential = fit.distribution_compare(
        "power_law", "exponential", normalized_ratio=True
    )
    return {
        "n_nodes_k>0": int(degrees.size),
        "alpha": fit.alpha,
        "x_min": fit.xmin,
        "n_tail": int(fit.n_tail),
        "R_vs_lognormal": r_lognormal,
        "p_vs_lognormal": p_lognormal,
        "R_vs_exponential": r_exponential,
        "p_vs_exponential": p_exponential,
        "_fit": fit,
    }

## Overview of all collections

In [14]:
overview_df = get_collections_overview(session)
overview_df

,collection,kind,n_entries,n_elements
0,PD_DM_CD,CellDesigner,40,15594
1,COVID_DM_CD,CellDesigner,22,8831
2,AD_KG_BEL,BEL,1,5024
3,COVID_KG_BEL,BEL,1,2796
4,PD_KG_BEL,BEL,1,2313


## CellDesigner disease-map collections

### Element counts

In [15]:
cd_stats_df = get_cd_collections_statistics(session)
cd_stats_df

,collection,species,reactions,modulations,compartments,distinct_hgnc_symbols
0,PD_DM_CD,6996,1561,2338,534,1385
1,COVID_DM_CD,3864,1221,174,136,802


### Species breakdown by class

In [16]:
cd_species_df = get_cd_species_breakdown(session)
cd_species_df

collection,COVID_DM_CD,PD_DM_CD,Total
species_class,,,
Protein,2449,3782,6231
Complex,573,1129,1702
SimpleMolecule,366,832,1198
Phenotype,110,610,720
RNA,160,374,534
Ion,62,198,260
Drug,80,46,126
Gene,64,25,89


## BEL knowledge-graph collections

### Node, subgraph and relationship counts

In [17]:
bel_stats_df = get_bel_collections_statistics(session)
bel_stats_df

,collection,n_nodes,n_subgraphs,n_relationships
0,AD_KG_BEL,5024,124,8896
1,COVID_KG_BEL,2796,1,2944
2,PD_KG_BEL,2313,66,3406


### Node breakdown by class

In [18]:
bel_nodes_df = get_bel_node_breakdown(session)
bel_nodes_df.head(20)  # full table written to CSV below

collection,AD_KG_BEL,COVID_KG_BEL,PD_KG_BEL,Total
node_class,,,,
Protein,1711,683,862,3256
Complex,798,799,146,1743
Abundance,476,549,220,1245
BiologicalProcess,446,171,299,916
Activity,549,87,137,773
Gene,219,103,242,564
Pathology,127,171,93,391
Variant,125,7,145,277
Rna,115,25,39,179


### Relationship breakdown by type

The BEL relation types grouped by family: **causal** (`INCREASES`, `DECREASES`,
their `DIRECTLY_*` variants, `REGULATES`, `CAUSES_NO_CHANGE`), **correlative**
(`ASSOCIATION`, `POSITIVE_CORRELATION`, `NEGATIVE_CORRELATION`), **hierarchical**
(`IS_A`), **equivalence** (`ORTHOLOGOUS`, `EQUIVALENT_TO`) and **transformation**
(`TRANSLATED_TO`). Only the causal family is carried into the influence graph by
`ig.make_ig_in_db`.

In [19]:
bel_relationships_df = get_bel_relationship_breakdown(session)
bel_relationships_df

collection,AD_KG_BEL,COVID_KG_BEL,PD_KG_BEL,Total
relation_type,,,,
INCREASES,4056,765,1476,6297
DECREASES,1598,868,594,3060
ASSOCIATION,1877,456,727,3060
POSITIVE_CORRELATION,528,183,307,1018
NEGATIVE_CORRELATION,335,141,182,658
REGULATES,155,251,9,415
IS_A,67,243,33,343
CAUSES_NO_CHANGE,58,9,60,127
DIRECTLY_INCREASES,114,6,6,126


### Relationship breakdown by endpoint type pair

Number of relationships for each *unordered* pair of endpoint node classes
(e.g. `Protein - Protein`, `BiologicalProcess - Protein`). The pair is
order-independent: an edge from a Protein to a BiologicalProcess and one in the
opposite direction are counted under the same `BiologicalProcess - Protein`
pair.

In [20]:
bel_type_pairs_df = get_bel_endpoint_type_pair_breakdown(session)
bel_type_pairs_df.head(20)  # full table written to CSV below

collection,AD_KG_BEL,COVID_KG_BEL,PD_KG_BEL,Total
type_pair,,,,
BiologicalProcess - Protein,1020,322,581,1923
Protein - Protein,996,198,549,1743
Abundance - Protein,922,342,195,1459
Pathology - Protein,779,154,364,1297
Activity - Protein,879,41,190,1110
Abundance - Pathology,289,614,75,978
Abundance - BiologicalProcess,376,228,119,723
Abundance - Activity,513,81,39,633
Abundance - Abundance,243,303,59,605


### Relationship breakdown by coarse endpoint group pair

The same relationships collapsed onto two coarse endpoint groups:
**abundance/activity** (abundance-type entities and their activities: `Protein`,
`Complex`, `Abundance`, `Gene`, `Rna`, `MicroRna`, `Composite`, `Variant`,
`Activity`) and **biologicalprocess/pathology** (the `BioConcept` nodes,
BiologicalProcess and Pathology). Every other node type is excluded, and only
relationships whose *both* endpoints are in these two groups are counted
(induced subgraph) -- so, unlike the tables above, these three rows do **not**
sum to the collection's total relationship count.

In [21]:
bel_group_pairs_df = get_bel_grouped_endpoint_pair_breakdown(session)
bel_group_pairs_df

collection,AD_KG_BEL,COVID_KG_BEL,PD_KG_BEL,Total
group_pair,,,,
abundance/activity - abundance/activity,4635,1216,1509,7360
abundance/activity - biologicalprocess/pathology,3400,1481,1574,6455
biologicalprocess/pathology - biologicalprocess/pathology,382,182,221,785


## Writing the statistics to disk

We write each table as a CSV under `RESULTS_DIR/collections_statistics/`.

In [22]:
COLLECTIONS_STATISTICS_DIR = RESULTS_DIR / "collections_statistics/"
commute_dm.utils.remake_dir(COLLECTIONS_STATISTICS_DIR)

overview_df.to_csv(COLLECTIONS_STATISTICS_DIR / "overview.csv", index=False)
cd_stats_df.to_csv(COLLECTIONS_STATISTICS_DIR / "cd_element_counts.csv", index=False)
cd_species_df.to_csv(COLLECTIONS_STATISTICS_DIR / "cd_species_breakdown.csv")
bel_stats_df.to_csv(COLLECTIONS_STATISTICS_DIR / "bel_node_counts.csv", index=False)
bel_nodes_df.to_csv(COLLECTIONS_STATISTICS_DIR / "bel_node_breakdown.csv")
bel_relationships_df.to_csv(
    COLLECTIONS_STATISTICS_DIR / "bel_relationship_breakdown.csv"
)
bel_type_pairs_df.to_csv(
    COLLECTIONS_STATISTICS_DIR / "bel_endpoint_type_pair_breakdown.csv"
)
bel_group_pairs_df.to_csv(
    COLLECTIONS_STATISTICS_DIR / "bel_grouped_endpoint_pair_breakdown.csv"
)

## Are the BEL KGs scale-free? Degree distributions and power-law fits

For each of the three BEL KGs we test whether its degree distribution is
**scale-free** (i.e. follows a power law `P(k) ~ k^-alpha`). We use the induced
counted subgraph (the **abundance/activity** and **biologicalprocess/pathology**
node groups only, and only relationships whose both endpoints are in that set)
and look at in-degree, out-degree and total degree separately.

Two ingredients, following Clauset, Shalizi & Newman (2009):

1. **Log-log CCDF** — the complementary cumulative distribution `P(K >= k)`,
   plotted on log-log axes (no binning artifacts). A straight line is the
   visual signature of a power law.
2. **A `powerlaw` fit** — MLE estimate of the exponent `alpha` and the lower
   cutoff `x_min`, plus normalised likelihood-ratio tests of the power law
   against a **lognormal** and an **exponential** alternative. `R > 0` favours
   the power law; `p` is the significance. A distribution is convincingly
   scale-free only if the power law is not rejected *and* is favoured over these
   alternatives (in practice a heavy tail is often statistically
   indistinguishable from a lognormal).

Note: `powerlaw` (and `scipy`) must be installed in the environment running this
notebook.

In [23]:
import warnings

import matplotlib.pyplot as plt

BEL_COLLECTIONS = ["AD_KG_BEL", "COVID_KG_BEL", "PD_KG_BEL"]
DEGREE_TYPES = [
    ("indegree", "in-degree"),
    ("outdegree", "out-degree"),
    ("total", "total degree"),
]

degree_df = get_bel_degree_data(session)
degree_df = degree_df.assign(total=degree_df["indegree"] + degree_df["outdegree"])

# Fit a power law to each (collection, degree-type) and collect the verdicts.
records = []
fits = {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # powerlaw is chatty about xmin / divide-by-zero
    for collection in BEL_COLLECTIONS:
        sub = degree_df[degree_df["collection"] == collection]
        for degree_col, label in DEGREE_TYPES:
            result = fit_degree_power_law(sub[degree_col].to_numpy())
            fits[(collection, degree_col)] = result.pop("_fit")
            records.append({"collection": collection, "degree": label, **result})

scale_free_df = pd.DataFrame(records)

# Log-log CCDFs with the fitted power-law tail overlaid (rows = KG, cols = degree type).
fig, axes = plt.subplots(
    len(BEL_COLLECTIONS), len(DEGREE_TYPES), figsize=(13, 11), squeeze=False
)
for i, collection in enumerate(BEL_COLLECTIONS):
    for j, (degree_col, label) in enumerate(DEGREE_TYPES):
        ax = axes[i][j]
        fit = fits[(collection, degree_col)]
        fit.plot_ccdf(
            ax=ax, color="tab:blue", marker=".", linestyle="none", label="empirical"
        )
        fit.power_law.plot_ccdf(
            ax=ax, color="tab:red", linestyle="--", label="power-law fit"
        )
        row = scale_free_df[
            (scale_free_df["collection"] == collection)
            & (scale_free_df["degree"] == label)
        ].iloc[0]
        ax.set_title(f"{collection} — {label}")
        ax.set_xlabel("degree k")
        ax.set_ylabel("P(K ≥ k)")
        ax.text(
            0.04,
            0.04,
            f"α = {row['alpha']:.2f}\nx_min = {row['x_min']:.0f}\n"
            f"vs exp: p={row['p_vs_exponential']:.3f}\n"
            f"vs lognorm: p={row['p_vs_lognormal']:.3f}",
            transform=ax.transAxes,
            va="bottom",
            ha="left",
            fontsize=8,
        )
        if i == 0 and j == 0:
            ax.legend(fontsize="small", loc="upper right")

fig.suptitle(
    "Degree-distribution CCDFs and power-law fits of the BEL KGs\n"
    "(induced on abundance/activity + biologicalprocess/pathology nodes)"
)
fig.tight_layout()
fig.savefig(
    COLLECTIONS_STATISTICS_DIR / "bel_degree_scalefree.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

scale_free_df.to_csv(
    COLLECTIONS_STATISTICS_DIR / "bel_scale_free_fits.csv", index=False
)
scale_free_df

,collection,degree,n_nodes_k>0,alpha,x_min,n_tail,R_vs_lognormal,p_vs_lognormal,R_vs_exponential,p_vs_exponential
0,AD_KG_BEL,in-degree,2374,2.382835,4.0,453,1.052094,0.292757,3.297206,0.000977
1,AD_KG_BEL,out-degree,2354,2.500160,7.0,235,-0.222599,0.823848,3.327694,0.000876
2,AD_KG_BEL,total degree,3581,2.480953,8.0,419,-1.000967,0.316843,3.385891,0.000709
3,COVID_KG_BEL,in-degree,895,2.272604,2.0,433,-0.754234,0.450708,2.215536,0.026723
4,COVID_KG_BEL,out-degree,849,2.874023,4.0,246,0.291002,0.771050,2.298864,0.021513
5,COVID_KG_BEL,total degree,1504,2.554900,4.0,426,-0.397679,0.690867,1.947730,0.051447
6,PD_KG_BEL,in-degree,1118,2.117065,1.0,1118,-1.201181,0.229681,2.503526,0.012296
7,PD_KG_BEL,out-degree,1181,2.353332,2.0,544,-0.701447,0.483024,3.356620,0.000789
8,PD_KG_BEL,total degree,1852,2.461906,6.0,217,0.214171,0.830414,2.364686,0.018045
